# Chapter 05 — Deep Learning

Notebook ini membangun pemahaman dari satu neuron sampai CNN, LSTM, dan transfer learning. Jalankan cell **berurutan**. TensorFlow diperlukan untuk bagian Keras; bagian NumPy tetap bisa dijalankan tanpa TensorFlow.

Tujuan utama bukan mengejar akurasi tertinggi, tetapi memahami alur: data → split → preprocessing → model → loss → optimisasi → evaluasi.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
    print(f'TensorFlow {tf.__version__} siap digunakan.')
except ImportError:
    HAS_TF = False
    print('TensorFlow belum terpasang. Jalankan: pip install tensorflow')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    if HAS_TF:
        tf.keras.utils.set_random_seed(seed)

set_seed()

## 1. Satu neuron dan gradient descent

Neuron linear menghitung `y_hat = w * x + b`. Loss MSE mengukur rata-rata kuadrat selisih `y_hat` dan `y`. Gradient descent memperbarui `w` dan `b` ke arah yang menurunkan loss. Perhatikan bahwa `x` dan `y` di bawah sama-sama array 1D; ini mencegah broadcasting yang tidak disengaja.

In [ ]:
x = np.arange(1, 11, dtype=np.float64)
y = 2 * x + np.array([0.1, -0.1, 0.2, -0.1, 0.0, 0.1, -0.2, 0.1, -0.1, 0.2])

weight, bias = 0.0, 0.0
learning_rate = 0.01
loss_history = []

for epoch in range(500):
    prediction = weight * x + bias
    error = prediction - y
    loss = np.mean(error ** 2)
    loss_history.append(loss)

    d_weight = 2 * np.mean(error * x)
    d_bias = 2 * np.mean(error)
    weight -= learning_rate * d_weight
    bias -= learning_rate * d_bias

print(f'y_hat = {weight:.3f} * x + {bias:.3f}')
print(f'Loss awal={loss_history[0]:.3f}; loss akhir={loss_history[-1]:.5f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x, y, label='data')
axes[0].plot(x, weight * x + bias, color='crimson', label='model')
axes[0].set(title='Regresi linear manual', xlabel='x', ylabel='y')
axes[0].legend()
axes[1].plot(loss_history)
axes[1].set(title='Loss selama training', xlabel='epoch', ylabel='MSE')
plt.tight_layout()

## 2. MLP untuk Fashion-MNIST

Fashion-MNIST memiliki gambar grayscale 28×28 dan 10 kelas. Nilai pixel perlu dinormalisasi ke 0–1. `SparseCategoricalCrossentropy` dipilih karena label tetap berupa integer, sehingga kita tidak perlu one-hot encode.

In [ ]:
if HAS_TF:
    (x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0
    class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                   'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    for image, label, ax in zip(x_train[:10], y_train[:10], axes.flat):
        ax.imshow(image, cmap='gray')
        ax.set_title(class_names[label])
        ax.axis('off')
    plt.tight_layout()
    print(f'Train: {x_train.shape}; test: {x_test.shape}')

In [ ]:
if HAS_TF:
    mlp = keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10, activation='softmax'),
    ])
    mlp.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy'],
    )
    mlp.summary()

In [ ]:
if HAS_TF:
    # Gunakan subset agar laptop pemula tetap responsif. Naikkan ke seluruh data setelah paham.
    train_limit, test_limit = 12_000, 2_000
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True
    )
    history_mlp = mlp.fit(
        x_train[:train_limit], y_train[:train_limit],
        validation_split=0.1, epochs=15, batch_size=128,
        callbacks=[early_stop], verbose=1,
    )
    mlp_loss, mlp_accuracy = mlp.evaluate(x_test[:test_limit], y_test[:test_limit], verbose=0)
    print(f'MLP test accuracy: {mlp_accuracy:.3f}')

## 3. CNN: memanfaatkan struktur spasial gambar

MLP meratakan gambar sehingga hubungan antar-pixel tetangga hilang. CNN memakai filter kecil yang digeser di gambar; `MaxPooling2D` mengecilkan peta fitur dan `Dropout` membantu regularisasi.

In [ ]:
if HAS_TF:
    cnn = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        keras.layers.MaxPooling2D(),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(10, activation='softmax'),
    ])
    cnn.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    cnn.summary()

In [ ]:
if HAS_TF:
    x_train_cnn = x_train[..., np.newaxis]
    x_test_cnn = x_test[..., np.newaxis]
    history_cnn = cnn.fit(
        x_train_cnn[:train_limit], y_train[:train_limit],
        validation_split=0.1, epochs=15, batch_size=128,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=3, restore_best_weights=True
        )],
        verbose=1,
    )
    cnn_loss, cnn_accuracy = cnn.evaluate(x_test_cnn[:test_limit], y_test[:test_limit], verbose=0)
    print(f'CNN test accuracy: {cnn_accuracy:.3f}')

In [ ]:
if HAS_TF:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_mlp.history['val_accuracy'], label='MLP')
    axes[0].plot(history_cnn.history['val_accuracy'], label='CNN')
    axes[0].set(title='Validation accuracy', xlabel='epoch', ylabel='accuracy')
    axes[0].legend()
    axes[1].bar(['MLP', 'CNN'], [mlp_accuracy, cnn_accuracy], color=['steelblue', 'darkorange'])
    axes[1].set(title='Akurasi pada test set', ylim=(0, 1))
    plt.tight_layout()

    predictions = np.argmax(cnn.predict(x_test_cnn[:test_limit], verbose=0), axis=1)
    confusion = tf.math.confusion_matrix(y_test[:test_limit], predictions).numpy()
    print('Confusion matrix (baris=aktual, kolom=prediksi):')
    print(confusion)

## 4. LSTM untuk data berurutan

Untuk time series, urutan waktu wajib dipertahankan: jangan mengacak data dan jangan memakai data masa depan. Normalisasi juga hanya dihitung dari training data.

In [ ]:
def make_windows(series, window):
    features = np.array([series[i:i + window] for i in range(len(series) - window)])
    target = series[window:]
    return features[..., np.newaxis], target

set_seed()
time = np.arange(500)
series = np.sin(time / 15) + 0.003 * time + np.random.normal(0, 0.1, len(time))
split_at = 400
mean_train, std_train = series[:split_at].mean(), series[:split_at].std()
series_scaled = (series - mean_train) / std_train
x_seq, y_seq = make_windows(series_scaled, window=30)
train_rows = split_at - 30
x_seq_train, y_seq_train = x_seq[:train_rows], y_seq[:train_rows]
x_seq_test, y_seq_test = x_seq[train_rows:], y_seq[train_rows:]

if HAS_TF:
    lstm = keras.Sequential([
        keras.layers.Input(shape=(30, 1)),
        keras.layers.LSTM(32),
        keras.layers.Dense(1),
    ])
    lstm.compile(optimizer='adam', loss='mse')
    lstm.fit(x_seq_train, y_seq_train, validation_split=0.1, epochs=20, verbose=0)
    forecast = lstm.predict(x_seq_test, verbose=0).ravel() * std_train + mean_train
    actual = y_seq_test * std_train + mean_train
    print(f'LSTM MAE: {np.mean(np.abs(forecast - actual)):.3f}')

## 5. Transfer learning dan checklist

Transfer learning memakai fitur dari model yang sudah dilatih pada dataset besar. Mulai dengan backbone dibekukan; fine-tuning hanya setelah baseline stabil, dengan learning rate kecil. Untuk data nyata, pisahkan train/validation/test sebelum augmentation, ukur metrik yang sesuai masalah, dan simpan konfigurasi/seed bersama model.

In [ ]:
if HAS_TF:
    # weights=None menjaga notebook dapat dibangun tanpa mengunduh bobot ImageNet.
    # Ganti menjadi weights='imagenet' saat koneksi internet tersedia.
    backbone = keras.applications.MobileNetV2(
        include_top=False, weights=None, input_shape=(160, 160, 3)
    )
    backbone.trainable = False
    transfer_model = keras.Sequential([
        keras.layers.Input(shape=(160, 160, 3)),
        backbone,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(2, activation='softmax'),
    ])
    transfer_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print('Model transfer learning siap. Hubungkan dengan dataset gambar milikmu.')

print('Eksperimen lanjutan: ubah learning rate, jumlah filter CNN, dan window LSTM; catat dampaknya.')